# LAB 2 — RAPTOR: Clustering Hierárquico e Árvore de Abstrações
## Aula 6: Indexação Avançada · MBA RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Objetivo:** Implementar o algoritmo RAPTOR sobre corpus jurídico — UMAP + GMM + sumarização recursiva — e visualizar a árvore de abstrações em 2D.

**Tempo estimado:** 50 minutos | **Pré-requisito:** Lab 1 concluído

## ⚙️ Passo 1 — Instalação

In [ ]:
!pip install umap-learn scikit-learn matplotlib sentence-transformers --quiet
!pip install langchain langchain-openai faiss-cpu --quiet
print('✅ Instalado')

## 📦 Passo 2 — Imports

In [ ]:
import json, os, warnings
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.mixture import GaussianMixture
import umap
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
warnings.filterwarnings('ignore')

try:
    encoder = SentenceTransformer('BAAI/bge-m3')
    print('BGE-M3 carregado')
except:
    encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    print('Fallback: MiniLM')

try:
    llm = ChatOpenAI(model='meta-llama/Llama-3.1-8B-Instruct', base_url='http://localhost:8000/v1', api_key='dummy', temperature=0.2, max_tokens=300)
    llm.invoke('ok')
    LLM_ON = True
    print('vLLM conectado')
except:
    LLM_ON = False
    print('vLLM offline - resumos simulados')

## 📚 Passo 3 — Corpus e Chunking

In [ ]:
try:
    with open('corpus_indexacao_avancada.json','r',encoding='utf-8') as f:
        c = json.load(f)
    raw = [(d['numero'], d['texto']) for d in c['documentos']]
except:
    raw = [
        ('HC 143641/STF','O STF determinou substituicao de prisao preventiva por domiciliar para gestantes e maes de criancas ate 12 anos. Lei 13.257/2016 alterou CPP.'),
        ('Lei 13.964/2019','Pacote Anticrime criou juiz das garantias e regulamentou cadeia de custodia (art 158-A CPP).'),
        ('RHC 143.308/STJ','Busca pessoal sem mandado requer fundada suspeita objetiva. Nervosismo nao e suficiente. Prova ilicita gera trancamento.'),
        ('RE 1.055.941/STF','COAF pode compartilhar dados com MP sem autorizacao judicial. Constitucional conforme Tema 990 STF.'),
        ('Parecer PGR 2023','Reconhecimento facial com vies nao pode ser prova direta. Recomenda protocolo nacional com auditoria.'),
        ('Decreto 11.531/2023','PNSP visa reduzir criminalidade integrar orgaos modernizar sistemas de informacao e fortalecer controle policial.'),
        ('Laudo INC 2024','847 transacoes Bitcoin 12.4 BTC rastreado para exchanges no exterior. Hash SHA-256 garantiu cadeia de custodia forense.'),
        ('Boletim SSP-SP 2023','Homicidios caíram 8.2% roubos de veiculos reduziram 18% apos expansao de cameras com reconhecimento de placas.'),
    ]

def chunk_text(text, size=60):
    words = text.split()
    return [' '.join(words[i:i+size]) for i in range(0, len(words), size)]

chunks, chunk_src = [], []
for num, txt in raw:
    for c in chunk_text(txt):
        chunks.append(c)
        chunk_src.append(num[:20])

print(f'{len(chunks)} chunks de {len(raw)} documentos')

## 🔢 Passo 4 — Embeddings

In [ ]:
print('Gerando embeddings...')
embeddings = encoder.encode(chunks, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
print(f'Shape: {embeddings.shape}')
norms = np.linalg.norm(embeddings, axis=1)
assert np.allclose(norms, 1.0, atol=1e-4)
print('Normalizacao: OK')

## 📉 Passo 5 — UMAP

In [ ]:
n_nb = min(5, len(chunks)-1)

# Para clustering
r_clust = umap.UMAP(n_components=min(10, len(chunks)-2), n_neighbors=n_nb, min_dist=0.0, metric='cosine', random_state=42)
emb_10d = r_clust.fit_transform(embeddings)
print(f'Clustering: {embeddings.shape[1]}D → {emb_10d.shape[1]}D')

# Para visualizacao
r_viz = umap.UMAP(n_components=2, n_neighbors=n_nb, min_dist=0.1, metric='cosine', random_state=42)
emb_2d = r_viz.fit_transform(embeddings)
print(f'Visualizacao: {embeddings.shape[1]}D → 2D')

## 🎯 Passo 6 — Clustering GMM

In [ ]:
max_k = min(len(chunks)-1, 6)
bic = []
for k in range(2, max_k+1):
    g = GaussianMixture(n_components=k, random_state=42)
    g.fit(emb_10d)
    bic.append(g.bic(emb_10d))

opt_k = list(range(2, max_k+1))[np.argmin(bic)]
print(f'k otimo: {opt_k} (menor BIC={min(bic):.0f})')

gmm = GaussianMixture(n_components=opt_k, random_state=42)
gmm.fit(emb_10d)
labels = gmm.predict(emb_10d)
probs = gmm.predict_proba(emb_10d)

for k in range(opt_k):
    n = np.sum(labels==k)
    print(f'  Cluster {k}: {n} chunks ({100*n/len(chunks):.0f}%)')

assert len(labels) == len(chunks)
print('Clustering: OK')

## 📊 Passo 7 — Visualização UMAP 2D

In [ ]:
colors = plt.cm.Set1(np.linspace(0, 1, opt_k))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: clusters coloridos
ax = axes[0]
for k in range(opt_k):
    m = labels == k
    ax.scatter(emb_2d[m,0], emb_2d[m,1], c=[colors[k]], s=80,
               label=f'Cluster {k} ({m.sum()})', alpha=0.8, edgecolors='w', lw=0.5)
    # Labels dos documentos
    for i, (x, y) in enumerate(emb_2d[m]):
        ax.annotate(chunk_src[np.where(m)[0][i]], (x,y), fontsize=5, alpha=0.5)
ax.set_title('Clusters RAPTOR — Corpus Juridico')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Plot 2: confianca do clustering
ax2 = axes[1]
sc = ax2.scatter(emb_2d[:,0], emb_2d[:,1], c=probs.max(axis=1), cmap='RdYlGn', s=80, alpha=0.9, edgecolors='w', lw=0.5)
plt.colorbar(sc, ax=ax2, label='Confianca')
ax2.set_title('Confianca GMM por Chunk')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('raptor_clusters.png', dpi=120)
plt.show()
print('Salvo: raptor_clusters.png')

## 📝 Passo 8 — Sumarização Recursiva

In [ ]:
def sumarizar(textos, cluster_id):
    combined = '\n---\n'.join(textos[:5])
    if LLM_ON:
        p = ChatPromptTemplate.from_messages([
            ('system','Voce e especialista juridico. Gere resumo tematico de 100 palavras dos excertos.'),
            ('human',f'Cluster {cluster_id}:\n{combined[:1500]}\nResumo:'),
        ])
        try:
            return (p|llm|StrOutputParser()).invoke({})
        except:
            pass
    # Fallback: primeiras frases
    frases = []
    for t in textos[:3]:
        f = t.split('.')[0]
        if f: frases.append(f.strip() + '.')
    return f'[Cluster {cluster_id}] ' + ' '.join(frases[:2])

print('Gerando resumos dos clusters (Nivel 1)...')
resumos = {}
for k in range(opt_k):
    textos_k = [chunks[i] for i in range(len(chunks)) if labels[i]==k]
    resumos[k] = sumarizar(textos_k, k)
    print(f'  Cluster {k}: {resumos[k][:80]}...')

# Nivel 2: sumarizar os resumos
raiz = sumarizar(list(resumos.values()), 'RAIZ')
print(f'\nRAIZ: {raiz[:120]}')

# Salvar indice RAPTOR
raptor_idx = {
    'nivel_0': [{'id':i,'texto':c,'cluster':int(labels[i])} for i,c in enumerate(chunks)],
    'nivel_1': [{'cluster_id':k,'resumo':resumos[k]} for k in range(opt_k)],
    'nivel_2': [{'id':'raiz','resumo':raiz}],
}

os.makedirs('resultados_aula6', exist_ok=True)
with open('resultados_aula6/raptor_index.json','w',encoding='utf-8') as f:
    json.dump(raptor_idx, f, ensure_ascii=False, indent=2)

print(f'\nArvore RAPTOR: {len(raptor_idx["nivel_0"])} chunks + {len(raptor_idx["nivel_1"])} resumos + 1 raiz')
print('Salvo: resultados_aula6/raptor_index.json')

## 🔍 Passo 9 — Collapsed Tree vs Tree Traversal

In [ ]:
# Indexar todos os niveis para busca
all_texts = ([item['texto'] for item in raptor_idx['nivel_0']] +
             [item['resumo'] for item in raptor_idx['nivel_1']] +
             [raptor_idx['nivel_2'][0]['resumo']])
all_meta = ([{'nivel':0} for _ in raptor_idx['nivel_0']] +
            [{'nivel':1} for _ in raptor_idx['nivel_1']] +
            [{'nivel':2}])

print('Gerando embeddings do indice RAPTOR...')
raptor_embs = encoder.encode(all_texts, normalize_embeddings=True, show_progress_bar=False)

def collapsed_search(query, top_k=3):
    q_e = encoder.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(raptor_embs, q_e)
    idx_top = np.argsort(scores)[::-1][:top_k]
    return [(scores[i], all_meta[i]['nivel'], all_texts[i]) for i in idx_top]

print('\n=== COLLAPSED TREE SEARCH ===')
for query in ['requisitos busca pessoal sem mandado', 'tendencias jurisprudencia seguranca publica']:
    print(f'\nQuery: {query}')
    for score, nivel, text in collapsed_search(query):
        print(f'  [Nivel {nivel}] score={score:.3f} | {text[:80]}...')
    print('  -> Query especifica prefere Nivel 0 (detalhes); abrangente prefere Nivel 1-2 (sintese)')

## 📚 Referências (ABNT)

SARTHI, P. et al. **RAPTOR: Recursive Abstractive Processing for Tree-Organized Retrieval**. In: *ICLR*, 2024. arXiv:2401.18059.

MCINNES, L.; HEALY, J. **UMAP: Uniform Manifold Approximation and Projection**. arXiv:1802.03426, 2018.

---
*MBA em RAG & CAG Aplicados a Direito e Segurança Pública — Aula 6, Lab 2*